# Chapter 5 — Exercises (Python version)

*Section 5.10 · Forecasting: Principles and Practice (2nd ed.) · solutions in R and Python*

Auto-generated from `ch05_exercises.Rmd` by `python/rmd_to_ipynb.py` — the R code lives in the .Rmd file. Run this notebook from its own folder.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath("../python"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import fpp
import fpp_ch05 as c5
plt.rcParams.update({"figure.figsize": (8, 4.2), "figure.dpi": 110})
# Only show real results: hide the matplotlib objects that plotting calls return
def _quiet_displayhook(value, _orig=sys.displayhook):
    import matplotlib.artist
    items = value if isinstance(value, (list, tuple, np.ndarray)) else [value]
    if len(items) and all(isinstance(i, matplotlib.artist.Artist) for i in np.ravel(np.asarray(items, dtype=object))):
        return
    _orig(value)
sys.displayhook = _quiet_displayhook
%matplotlib inline

> **Acknowledgement.** Exercises from Section 5.10 of *Forecasting: Principles and Practice*
> (2nd ed.) by **Rob J Hyndman and George Athanasopoulos** (<https://otexts.com/fpp2/>).
> The questions are paraphrased; the answers are my own.

# Exercise 1 — Electricity demand and temperature (first 20 days of 2014)

*Using `daily20 <- head(elecdaily, 20)`: (a) plot the data and regress Demand on Temperature — why is the
relationship positive? (b) residual plot: is the model adequate, any outliers/influential points?
(c) forecast demand for a day with maximum temperature 15°C and 35°C — believable? (d) prediction
intervals; (e) plot Demand vs Temperature for all of `elecdaily` — what does that say about the model?*

### (a) Plot and fit

In [ ]:
elec = fpp.load("elecdaily")
daily20 = elec.iloc[:20]; daily20.attrs = dict(elec.attrs)
fpp.autoplot(daily20, facets=True); plt.show()
import seaborn as sns
sns.regplot(data=daily20, x="Temperature", y="Demand", ci=None); plt.show()
y20 = daily20["Demand"]; y20.attrs = dict(elec.attrs)
fit20 = c5.tslm(y20, X=daily20[["Temperature"]])
fit20.params

**Answer.** These are the first 20 days of January — Australian **summer**. Hotter days mean more air
conditioning, so demand rises by about 6.8 GW for each extra degree.

### (b) Residuals

In [ ]:
ax_ = c5.checkresiduals_lm(fit20)
plt.scatter(fit20.fitted, fit20.residuals); plt.xlabel("Fitted"); plt.ylabel("Residuals"); plt.show()
cooks = fit20.res.get_influence().cooks_distance[0]
pd.Series(cooks, index=np.arange(1, 21)).sort_values(ascending=False).head(3).round(3)

**Answer.** With only 20 points the BG test has little power, but the residuals are reasonably random.
One or two hot days (around 40°C) have large residuals and the largest Cook's distances — they are
influential, pulling the line. The model is acceptable for this narrow temperature range only.

### (c)–(d) Forecasts at 15°C and 35°C with prediction intervals

In [ ]:
fc20 = c5.forecast_lm(fit20, newdata=pd.DataFrame({"Temperature": [15.0, 35.0]}))
print(fc20.summary().round(3))
(daily20["Temperature"].min(), daily20["Temperature"].max())

**Answer.** 35°C (≈276 GW) is inside the observed temperature range, so it is plausible. 15°C (≈141 GW)
is far below anything in these 20 days; extrapolating a straight line there is not credible (on a cool
day heating, not cooling, drives demand). The 95% intervals are wide (about ±50 GW).

### (e) All of 2014

In [ ]:
plt.scatter(elec.Temperature, elec.Demand, s=8, color="black")
plt.scatter(daily20.Temperature, daily20.Demand, s=12, color="red")
plt.xlabel("Temperature"); plt.ylabel("Demand"); plt.show()

**Answer.** Over the whole year the relationship is **U-shaped** (high demand on both cold and hot
days). The 20-day linear model only describes the hot part; it would badly over-predict at 15°C — a
nonlinear (e.g. quadratic or piecewise) term in temperature is needed, as in Chapter 1's notes.

# Exercise 2 — Men's 400 m Olympic winning times

*(a) Plot and describe; (b) fit a regression line — average decrease per year? (c) plot residuals
against year; (d) predict 2020 with a prediction interval — what assumptions?*

### Solution

In [ ]:
mens = fpp.load("mens400")
fpp.autoplot(mens, xlab="Year", ylab="Winning time (seconds)"); plt.show()
fit400 = c5.tslm(mens, trend_=True)
print({"per_olympiad": fit400.params["trend"], "per_year": fit400.params["trend"] / 4})
plt.plot(fpp.ts_time(mens), fit400.residuals, marker="o"); plt.ylabel("Residuals"); plt.show()
c5.forecast_lm(fit400, h=1).summary()

**Answer.** (a) Times fall steeply early on (54.2 s in 1896) then level off around 43–44 s; there are
gaps for 1916, 1940 and 1944 (no Games). (b) The fitted line falls 0.258 s per Olympiad, i.e. about
**0.065 seconds per year**. (c) The residuals are U-shaped (positive at both ends, negative in the
middle): the trend is not linear, so a straight line is a poor model. (d) The 2020 forecast is 42.04 s
(95% interval roughly 39.6–44.5 s). This assumes the linear decline continues and that errors are
independent, normal and of constant variance — all doubtful given (c); the actual 2020 (2021) winning
time was 43.85 s, near the top of the interval.

# Exercise 3 — `easter(ausbeer)`

### Solution

In [ ]:
ea = c5.easter(fpp.load("ausbeer"))
ea.head(24).to_numpy().reshape(-1, 4)          # rows = years 1956.., columns = Q1..Q4

**Answer.** It is a quarterly Easter dummy: 1 in the quarter containing the Easter holidays
(Good Friday to Easter Sunday) — Q1 if Easter is in March, Q2 if in April — and 0 otherwise. When the
three-day holiday straddles 31 March, the value is split between Q1 and Q2 (e.g. 1956: 0.67 / 0.33,
1961: 0.33 / 0.67). Python's `c5.easter()` reproduces R's values exactly.

# Exercise 4 — Elasticity in the log-log model

*Show that in $\log y=\beta_0+\beta_1\log x+\varepsilon$, $\beta_1$ is the elasticity $(dy/dx)(x/y)$.*

**Derivation.** Exponentiating, $y = e^{\beta_0}x^{\beta_1}e^{\varepsilon}$. Ignoring the error,
$\frac{dy}{dx} = \beta_1 e^{\beta_0}x^{\beta_1-1} = \beta_1\frac{y}{x}$, so
$\frac{dy}{dx}\cdot\frac{x}{y} = \beta_1$. (Equivalently $\beta_1 = \frac{d\log y}{d\log x}$.)
The elasticity is constant: a 1% change in $x$ gives a $\beta_1$% change in $y$ everywhere.

### Numerical check with real data

My check: UK drivers killed vs. petrol price (`Seatbelts`) — the finite-difference elasticity equals $\hat\beta_1$.

In [ ]:
sb = fpp.load("Seatbelts")
fit_ll = smf.ols("np.log(drivers) ~ np.log(PetrolPrice)", data=sb).fit()
b0, b1 = fit_ll.params
yfun = lambda x: np.exp(b0) * x ** b1
x0, dx = sb.PetrolPrice.mean(), 1e-6
{"beta1": b1, "elasticity": (yfun(x0 + dx) - yfun(x0)) / dx * x0 / yfun(x0)}

A 1% rise in the real petrol price goes with about a 0.67% fall in drivers killed or injured.

# Exercise 5 — Souvenir shop sales (`fancy`)

### (a) Time plot

In [ ]:
fancy = fpp.load("fancy")
fpp.autoplot(fancy, ylab="Sales (A$)", xlab="Year"); plt.show()

**Answer.** Strong upward trend and seasonality whose size **grows with the level** (multiplicative).
Huge December peaks (Christmas) and a smaller peak in March from 1988 (the surfing festival). Sales in
March 1987 had no festival peak; there are some irregular jumps (e.g. late 1990–1991 looks weaker).

**(b)** Logs are needed because the seasonal fluctuations and variance increase with the level; on the
log scale they become roughly constant, so a model with *additive* trend and seasonal dummies fits, and
the effects become percentage effects.

### (c) Log model with trend, seasonal dummies and a surfing-festival dummy

In [ ]:
surf = ((fancy.index.month == 3) & (fancy.index.year >= 1988)).astype(float)
logf = np.log(fancy); logf.attrs = dict(fancy.attrs)
fit_fancy = c5.tslm(logf, trend_=True, season=True, X=pd.DataFrame({"surf": surf}))
print(fit_fancy.res.summary().tables[1])

### (d)–(e) Residual plots and monthly boxplots

In [ ]:
r = fit_fancy.residuals
plt.plot(fpp.ts_time(fancy), r); plt.ylabel("Residuals"); plt.show()
plt.scatter(fit_fancy.res.fittedvalues, r); plt.xlabel("Fitted (log scale)"); plt.ylabel("Residuals"); plt.show()
months = [pd.Period(year=2000, month=i, freq="M").strftime("%b") for i in range(1, 13)]
plt.boxplot([r[fancy.index.month == i].to_numpy() for i in range(1, 13)], tick_labels=months)
plt.ylabel("Residuals"); plt.show()

**Answer.** The residuals against time show a wave: positive mid-sample, negative around 1990–91 and
rising at the end — leftover autocorrelation / a trend that is not linear. Against the fitted values they
look roughly patternless (logs fixed the variance). The monthly boxplots show different spreads by month
(and medians not all at zero, e.g. some autumn months), so the seasonal pattern is not perfectly
captured.

### (f)–(g) Coefficients and Breusch–Godfrey

In [ ]:
print((np.exp(fit_fancy.params) - 1).round(3))
c5.bgtest(fit_fancy, order=24)

**Answer.** On the log scale each coefficient is (approximately) a percentage effect: the trend means
sales grow about 2.2% per month; every month exceeds January, with December about 7 times
January's level (+611%) and November more than 3 times; the surf dummy adds about 65% to March sales
from 1988 (on top of March's normal seasonal effect). The BG test rejects
(p < 0.05): the residuals are autocorrelated, so prediction intervals are too narrow and the forecasts
are not using all the information.

### (h)–(i) Forecasts for 1994–1996, then back to the original scale

In [ ]:
future_surf = pd.DataFrame({"surf": [0, 0, 1] + [0] * 9 + [0, 0, 1] + [0] * 9 + [0, 0, 1] + [0] * 9})
fc_log = c5.forecast_lm(fit_fancy, newdata=future_surf)
print(fc_log.summary().head())
fc_raw = fpp.Forecast("Back-transformed regression", fancy, np.exp(fc_log.mean), np.exp(fc_log.lower),
                      np.exp(fc_log.upper), fancy * np.nan, fancy * np.nan)
fpp.plot_forecast(fc_raw, ylab="Sales (A$)"); plt.show()
pd.DataFrame({"mean": fc_raw.mean, "lo95": fc_raw.lower[95], "hi95": fc_raw.upper[95]}).head()

**(j) Improvements.** Model the autocorrelation (regression with ARIMA errors, Chapter 9); a
piecewise/nonlinear trend to capture the 1990–91 slowdown; Easter and trading-day predictors; use the
bias-adjusted back-transformation (the exponentiated forecast is the *median*, not the mean); and
check that the surfing festival is still held each March.

# Exercise 6 — Harmonic regression for weekly gasoline supplies (to end of 2004)

### (a)–(b) Choose the number of Fourier terms by AICc / CV

In [ ]:
gas = fpp.window(fpp.load("gasoline"), end=2005)
crit = pd.DataFrame([{"K": K, **c5.CV(c5.tslm(gas, trend_=True, K=K))} for K in range(1, 27)])
print(crit.iloc[[0, 1, 2, 5, 6, 7, 8, 9, 10, 11, 12, 13, 25]].round(5))
print({"best_AICc": int(crit.K[crit.AICc.idxmin()]), "best_CV": int(crit.K[crit.CV.idxmin()])})
tg = fpp.ts_time(gas)
fig, axes = plt.subplots(3, 1, figsize=(8, 8), sharex=True)
for a, K in zip(axes, [1, 5, 12]):
    a.plot(tg, gas, lw=0.6, label="Data"); a.plot(tg, c5.tslm(gas, trend_=True, K=K).fitted, label="Fitted")
    a.set_title(f"Harmonic regression, K = {K}")
axes[0].legend(); plt.tight_layout(); plt.show()

**Answer.** With K = 1 the fitted seasonal curve is a single smooth wave; more terms add the summer
peak's sharper shape. AICc and CV are both minimised at **K = 12**. The fitted values follow the seasonal
pattern but not the medium-term swings in the trend (e.g. the flattening after 2003).

### (c) Residuals of the final model

In [ ]:
fit_gas = c5.tslm(gas, trend_=True, K=12)
ax_ = c5.checkresiduals_lm(fit_gas)

The BG test rejects, but the autocorrelations are small; the forecasts will hardly change.

### (d)–(e) Forecast 2005 and compare with the actual data

In [ ]:
fc_gas = c5.forecast_lm(fit_gas, h=52)
actual05 = fpp.window(fpp.load("gasoline"), start=2005.01, end=2006)
ax = fpp.plot_forecast(fc_gas, include=150, ylab="Million barrels per day")
ax.plot(fpp.ts_time(actual05), actual05, color="red", label="Actual 2005"); ax.legend(); plt.show()
print(fpp.accuracy(fc_gas, actual05).loc["Test set", ["RMSE", "MAPE"]])
float(np.mean((actual05.to_numpy() >= fc_gas.lower[95].to_numpy()) & (actual05.to_numpy() <= fc_gas.upper[95].to_numpy())))

**Answer.** The forecasts follow the 2005 seasonal pattern well (MAPE about 1.7%) and 51 of the 52
actual weeks (98%) lie inside the 95% prediction intervals — the small leftover autocorrelation did not
matter much, as the exercise suggested. The largest misses are individual weeks of unusual supply that no
seasonal pattern can anticipate.

# Exercise 7 — Lake Huron water level, 1875–1972

### Solution

In [ ]:
huron = fpp.load("huron")
th = fpp.ts_time(huron)
fit_lin = c5.tslm(huron, trend_=True)
fit_pw = c5.tslm(huron, X=pd.DataFrame({"t": th, "t15": np.maximum(0, th - 1915)}))
fc_lin = c5.forecast_lm(fit_lin, h=8)
fc_pw = c5.forecast_lm(fit_pw, newdata=pd.DataFrame({"t": np.arange(1973, 1981.0), "t15": np.arange(1973, 1981.0) - 1915}))
fig, ax = plt.subplots()
ax.plot(th, huron, color="black")
for f, fc, lab in [(fit_lin, fc_lin, "Linear"), (fit_pw, fc_pw, "Piecewise")]:
    line, = ax.plot(th, f.fitted, label=lab); ax.plot(np.arange(1973, 1981), fc.mean, color=line.get_color())
ax.legend(); ax.set_ylabel("Level (feet, minus 570)"); plt.show()
print(pd.DataFrame({"linear": c5.CV(fit_lin), "piecewise": c5.CV(fit_pw)}).T.round(3))
pd.DataFrame({"linear": fc_lin.mean.to_numpy(), "piecewise": fc_pw.mean.to_numpy()}, index=range(1973, 1981))

**Answer.** (a) The level falls until about 1915–1930 and then fluctuates with no clear trend, with
long, autocorrelated swings. (b) The piecewise model (slope changes in 1915) fits better (lower AICc and
CV). (c) The linear trend keeps forecasting a decline to 1980; the piecewise trend is almost flat after
1915, giving forecasts near the recent average — more sensible, although neither captures the long swings.

# Exercise 8 — Matrix algebra for the trend model $y_t=\beta_0+\beta_1 t$

With $\sum t = \frac{T(T+1)}{2}$ and $\sum t^2 = \frac{T(T+1)(2T+1)}{6}$,
$\boldsymbol X'\boldsymbol X=\begin{pmatrix}T & \frac{T(T+1)}2\\ \frac{T(T+1)}2 & \frac{T(T+1)(2T+1)}6\end{pmatrix}$,
whose determinant is $\frac{T^2(T+1)(T-1)}{12}$. Hence

(a) $(\boldsymbol X'\boldsymbol X)^{-1}=\frac{2}{T(T-1)}\begin{pmatrix}2T+1 & -3\\ -3 & \frac{6}{T+1}\end{pmatrix}$;

(b) $\hat\beta_0=\frac{2}{T(T-1)}\Big[(2T+1)\sum y_t-3\sum t y_t\Big]$,
$\hat\beta_1=\frac{6}{T(T^2-1)}\Big[2\sum t y_t-(T+1)\sum y_t\Big]$;

(c) $\hat\sigma^2=\frac{1}{T-2}\sum (y_t-\hat\beta_0-\hat\beta_1 t)^2$;

(d) with $\boldsymbol x^*=(1,\,T+h)$:
$\mathrm{Var}(\hat y_{T+h})=\hat\sigma^2\Big[1+\frac{2}{T(T-1)}\Big(1-4T-6h+6\frac{(T+h)^2}{T+1}\Big)\Big]$,
since $\boldsymbol x^*(\boldsymbol X'\boldsymbol X)^{-1}\boldsymbol x^{*\prime}=\frac{2}{T(T-1)}\big[(2T+1)-6(T+h)+\frac{6(T+h)^2}{T+1}\big]$.

### Numerical check of every formula (Lake Huron data)

In [ ]:
y = huron.to_numpy(); T = len(y); tt = np.arange(1, T + 1); h = 5
X = np.column_stack([np.ones(T), tt])
XtXi_formula = 2 / (T * (T - 1)) * np.array([[2 * T + 1, -3], [-3, 6 / (T + 1)]])
b0 = 2 / (T * (T - 1)) * ((2 * T + 1) * y.sum() - 3 * (tt * y).sum())
b1 = 6 / (T * (T ** 2 - 1)) * (2 * (tt * y).sum() - (T + 1) * y.sum())
beta = np.linalg.lstsq(X, y, rcond=None)[0]
sigma2 = np.sum((y - X @ beta) ** 2) / (T - 2)
var_formula = sigma2 * (1 + 2 / (T * (T - 1)) * (1 - 4 * T - 6 * h + 6 * (T + h) ** 2 / (T + 1)))
xs = np.array([1, T + h])
var_matrix = sigma2 * (1 + xs @ np.linalg.inv(X.T @ X) @ xs)
{"max |XtXinv - formula|": float(np.abs(np.linalg.inv(X.T @ X) - XtXi_formula).max()),
 "beta formula - lstsq": (b0 - beta[0], b1 - beta[1]),
 "var_formula": var_formula, "var_matrix": var_matrix}

All differences are at rounding level (≈1e-15), and the variance formula reproduces the width of R's
prediction interval exactly.

*Reference: Hyndman, R.J., & Athanasopoulos, G. (2018). Forecasting: Principles and Practice
(2nd ed.). OTexts. <https://otexts.com/fpp2/>*